# EDA — Análisis exploratorio del dataset ISIC 2019 (Fase 2 CRISP-DM, Sprint 1)

**Proyecto:** DERM_IAPP — aplicación móvil con Deep Learning para detección de carcinoma
basocelular (UPAO, Trujillo 2026).

## Propósito

Este notebook es el **complemento reproducible de la Fase 2 (Entendimiento de los datos) de
CRISP-DM**. Verifica, con código ejecutable y asserts, cada cifra del entendimiento de datos:
composición de clases, vista binaria BCC/no-BCC, calidad de metadatos, estructura de
`lesion_id`, demografía por clase, proporción train/test y fuga por identificador. **No
construye el pool, ni el marco muestral, ni el split**: eso corresponde a los notebooks
`03_preparacion_datos.ipynb` y `04_marco_evaluacion.ipynb`.

## Fuentes de datos (solo 4 CSV oficiales)

| Archivo | Tamaño (bytes) |
|---|---:|
| `ISIC_2019_Training_Metadata.csv` | 1 214 351 |
| `ISIC_2019_Training_GroundTruth.csv` | 1 291 479 |
| `ISIC_2019_Test_Metadata.csv` | 293 638 |
| `ISIC_2019_Test_GroundTruth.csv` | 464 810 |

Los CSV se descargan de los enlaces S3 estables del README de `PRODUCTO/DATA/` y se verifican
**byte a byte** antes de usarse (§8.1 del análisis).

## Nota de vigencia

- **Vigentes:** §1–§7 de `PRODUCTO/DATA/ANALISIS_DATASET_ISIC2019.md` — son la referencia de
  entendimiento de datos y la fuente de los valores esperados de este notebook.
- **Obsoletos:** §8.4, §8.5 y el anexo de ese documento, que describen el **marco muestral v1**
  (60 subconjuntos de solo-test, techo de BCC = 975). Fueron sustituidos por el **marco
  muestral v2**, documento normativo en `PRODUCTO/DERM_IAPP/model/docs/marco_muestral_v2.md`
  (90 subconjuntos en 3 familias TA/NFS/PEO sobre el pool fusionado; techo de BCC = 4 298).

## Qué produce

Tablas (`.csv`), un resumen estructurado (`.json`) y, si `matplotlib` está disponible, 3
gráficos (`.png`) en `/content/derm_iapp/salidas_eda/`.

## Requisitos

- **No requiere los ZIP de imágenes**: todo el EDA trabaja sobre metadatos y etiquetas.
- Entorno: Google Colab (Python 3). Las dependencias que falten se instalan solas.
- Semilla del proyecto: `SEED = 2026` (el EDA no sortea nada; se declara por convención).


In [ ]:
# =============================================================================
# Celda 1 — Entorno mínimo, semilla y rutas base (§8.1)
# =============================================================================
import subprocess
import sys


def asegurar_paquete(modulo, paquete=None):
    """Instala con pip solo si el import del módulo falta (Colab recién creado)."""
    paquete = paquete or modulo
    try:
        __import__(modulo)
    except ImportError:
        print(f"[setup] Instalando {paquete} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])


# Dependencias obligatorias del EDA (solo metadatos: no se necesitan los ZIP)
for _modulo, _paquete in (("pandas", "pandas"), ("numpy", "numpy"), ("requests", "requests")):
    asegurar_paquete(_modulo, _paquete)

# matplotlib es OPCIONAL: si falta, la celda 11 omite los gráficos sin romper el notebook
try:
    import matplotlib

    matplotlib.use("Agg")  # backend sin ventana (Colab / servidor)
    import matplotlib.pyplot as plt

    HAY_MATPLOTLIB = True
except ImportError:
    plt = None
    HAY_MATPLOTLIB = False
    print("[setup] matplotlib no disponible: se omitirán los gráficos de la celda 11.")

import json
import os

import numpy as np
import pandas as pd

# Semilla global del proyecto: el EDA no sortea nada, se declara por convención
SEED = 2026

# Rutas base en Google Colab (§8.1): los CSV se leen de la raíz y las salidas van a salidas_eda/
BASE_DIR = "/content/derm_iapp"
SALIDAS_DIR = os.path.join(BASE_DIR, "salidas_eda")
os.makedirs(SALIDAS_DIR, exist_ok=True)

# Los 4 CSV oficiales con su tamaño byte-exacto del inventario (§8.1)
CSV_OFICIALES = {
    "ISIC_2019_Training_Metadata.csv": {
        "bytes": 1214351,
        "url": "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Metadata.csv",
    },
    "ISIC_2019_Training_GroundTruth.csv": {
        "bytes": 1291479,
        "url": "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_GroundTruth.csv",
    },
    "ISIC_2019_Test_Metadata.csv": {
        "bytes": 293638,
        "url": "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Test_Metadata.csv",
    },
    "ISIC_2019_Test_GroundTruth.csv": {
        "bytes": 464810,
        "url": "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Test_GroundTruth.csv",
    },
}

# Las 9 clases del desafío, en el orden de las columnas one-hot del GroundTruth (§2)
CLASES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC", "UNK"]

print(f"[setup] SEED={SEED} | BASE_DIR={BASE_DIR} | salidas -> {SALIDAS_DIR}")
print(f"[setup] matplotlib disponible: {HAY_MATPLOTLIB}")


In [ ]:
# =============================================================================
# Celda 2 — Descarga autónoma de los 4 CSV con verificación byte-exacta (§8.1)
# Regla R1 del marco v2; links del README de PRODUCTO/DATA/.
# =============================================================================


def asegurar_csv(nombre, url, tam):
    """Garantiza que el CSV existe en BASE_DIR con el tamaño byte-exacto esperado.

    - Existe y pesa igual  -> no se re-descarga (idempotente).
    - Existe con otro peso -> error duro (archivo corrupto o versión distinta).
    - No existe            -> descarga por streaming y verifica el tamaño final.
    """
    ruta = os.path.join(BASE_DIR, nombre)

    if os.path.exists(ruta):
        actual = os.path.getsize(ruta)
        if actual == tam:
            print(f"[CSV] OK  {nombre}: ya existe con {actual} B (no se re-descarga)")
            return ruta
        raise RuntimeError(
            f"[CSV] Archivo corrupto o de versión distinta: {nombre}. "
            f"Esperado {tam} B, obtenido {actual} B. Bórralo y vuelve a ejecutar la celda."
        )

    print(f"[CSV] Descargando {nombre} ({tam} B) ...")
    with requests.get(url, stream=True, timeout=180) as respuesta:
        respuesta.raise_for_status()
        with open(ruta, "wb") as destino:
            for bloque in respuesta.iter_content(chunk_size=1 << 20):  # bloques de 1 MiB
                if bloque:
                    destino.write(bloque)

    actual = os.path.getsize(ruta)
    if actual != tam:
        os.remove(ruta)
        raise RuntimeError(
            f"[CSV] Descarga incompleta de {nombre}: esperado {tam} B, obtenido {actual} B. "
            "Se eliminó el archivo; revisa la conexión y reintenta."
        )

    print(f"[CSV] OK  {nombre}: descargado con {actual} B")
    return ruta


RUTAS_CSV = {
    nombre: asegurar_csv(nombre, meta["url"], meta["bytes"])
    for nombre, meta in CSV_OFICIALES.items()
}
assert len(RUTAS_CSV) == 4, f"Se esperaban 4 CSV oficiales, se resolvieron {len(RUTAS_CSV)}"
print("\n[CSV] Los 4 CSV oficiales están disponibles y verificados por tamaño.")


In [ ]:
# =============================================================================
# Celda 3 — Carga y derivación de etiquetas (§2)
# El GroundTruth es one-hot de 9 columnas: exactamente un 1.0 por fila.
# =============================================================================
COLUMNAS_ONE_HOT = CLASES  # nombres de las 9 columnas binarias del GroundTruth

meta_train = pd.read_csv(RUTAS_CSV["ISIC_2019_Training_Metadata.csv"])
gt_train = pd.read_csv(RUTAS_CSV["ISIC_2019_Training_GroundTruth.csv"])
meta_test = pd.read_csv(RUTAS_CSV["ISIC_2019_Test_Metadata.csv"])
gt_test = pd.read_csv(RUTAS_CSV["ISIC_2019_Test_GroundTruth.csv"])

print("[§2] GroundTruth train:", gt_train.shape, "| GroundTruth test:", gt_test.shape)
print("[§2] Metadata   train:", meta_train.shape, "| Metadata   test:", meta_test.shape)


def derivar_etiqueta(ground_truth, particion):
    """Convierte el one-hot de 9 clases en `clase_9` (str) y `es_bcc` (bool), con verificación."""
    faltantes_cols = [c for c in COLUMNAS_ONE_HOT if c not in ground_truth.columns]
    assert not faltantes_cols, (
        f"§2: faltan columnas one-hot en el GroundTruth de {particion}: {faltantes_cols}"
    )

    matriz = ground_truth[COLUMNAS_ONE_HOT].to_numpy(dtype="float64")
    suma_por_fila = matriz.sum(axis=1)
    n_multi = int((suma_por_fila > 1.0).sum())
    n_vacias = int((suma_por_fila == 0.0).sum())
    assert n_multi == 0, (
        f"§2: {particion} tiene {n_multi} filas multi-etiqueta; se esperaban 0"
    )
    assert n_vacias == 0, (
        f"§2: {particion} tiene {n_vacias} filas sin etiqueta; se esperaban 0"
    )

    resultado = ground_truth.copy()
    # Con exactamente un 1.0 por fila, argmax identifica la clase sin ambigüedad
    resultado["clase_9"] = [COLUMNAS_ONE_HOT[i] for i in matriz.argmax(axis=1)]
    resultado["es_bcc"] = resultado["clase_9"] == "BCC"
    return resultado


# En el test el GroundTruth trae 2 columnas extra (pesos del desafío): se IGNORAN en la tesis
assert {"score_weight", "validation_weight"}.issubset(gt_test.columns), (
    "§2: se esperaban las columnas score_weight y validation_weight en el GroundTruth del test "
    f"(no se usan); columnas actuales: {list(gt_test.columns)}"
)

gt_train = derivar_etiqueta(gt_train, "train")
gt_test = derivar_etiqueta(gt_test, "test")

# Tamaños oficiales (§1)
assert len(gt_train) == 25331, f"§1: filas de train esperadas 25331, obtenidas {len(gt_train)}"
assert len(gt_test) == 8238, f"§1: filas de test esperadas 8238, obtenidas {len(gt_test)}"

# Unión metadata + groundtruth por `image` en cada partición
train = meta_train.merge(gt_train, on="image", how="inner")
test = meta_test.merge(gt_test, on="image", how="inner")
assert len(train) == 25331, (
    f"§2: unión train inconsistente: filas esperadas 25331, obtenidas {len(train)}"
)
assert len(test) == 8238, f"§2: unión test inconsistente: filas esperadas 8238, obtenidas {len(test)}"

# §4.3: el metadata del test NO trae lesion_id (limitación estructural del dataset oficial)
assert "lesion_id" not in meta_test.columns, (
    "§4.3: el metadata del test no debe contener la columna lesion_id; se encontró"
)

print("[§2] Etiqueta derivada: 0 filas multi-etiqueta y 0 filas sin etiqueta en train y test.")
print("[§4.3] Confirmado: el metadata del test NO contiene la columna lesion_id.")


In [ ]:
# =============================================================================
# Celda 4 — Composición de las 9 clases en train y test (§3, §4)
# =============================================================================
# §3: composición del train (valores esperados)
ESPERADO_TRAIN = {
    "NV": 12875,
    "MEL": 4522,
    "BCC": 3323,
    "BKL": 2624,
    "AK": 867,
    "SCC": 628,
    "VASC": 253,
    "DF": 239,
}
# §4: composición del test (valores esperados)
ESPERADO_TEST = {
    "NV": 2495,
    "UNK": 2047,
    "MEL": 1327,
    "BCC": 975,
    "BKL": 660,
    "AK": 374,
    "SCC": 165,
    "VASC": 104,
    "DF": 91,
}

comp_train = train["clase_9"].value_counts()
comp_test = test["clase_9"].value_counts()

for clase, esperado in ESPERADO_TRAIN.items():
    obtenido = int(comp_train.get(clase, 0))
    assert obtenido == esperado, (
        f"§3: composición del train — {clase}: esperado {esperado}, obtenido {obtenido}"
    )
for clase, esperado in ESPERADO_TEST.items():
    obtenido = int(comp_test.get(clase, 0))
    assert obtenido == esperado, (
        f"§4: composición del test — {clase}: esperado {esperado}, obtenido {obtenido}"
    )

# UNK es una clase del desafío que solo aparece en el test (§2, §4)
unk_train = int(comp_train.get("UNK", 0))
assert unk_train == 0, f"§3: se esperaban 0 UNK en el train, se obtuvieron {unk_train}"

composicion = pd.DataFrame(
    {
        "clase": CLASES,
        "n_train": [int(comp_train.get(c, 0)) for c in CLASES],
        "n_test": [int(comp_test.get(c, 0)) for c in CLASES],
    }
)
composicion["pct_train"] = (composicion["n_train"] / len(train) * 100).round(2)
composicion["pct_test"] = (composicion["n_test"] / len(test) * 100).round(2)

print("[§3, §4] Composición de las 9 clases (cada cifra verificada con assert):")
print(composicion.to_string(index=False))

ruta_composicion = os.path.join(SALIDAS_DIR, "composicion_clases.csv")
composicion[["clase", "n_train", "n_test"]].to_csv(ruta_composicion, index=False)
print(f"\n[guardado] {ruta_composicion}")


In [ ]:
# =============================================================================
# Celda 5 — Vista binaria BCC / no-BCC (§3, §4)
# =============================================================================
# --- Train completo: BCC 3 323 (13,12 %) vs no-BCC 22 008 (86,88 %) (§3)
n_bcc_train = int(train["es_bcc"].sum())
n_nobcc_train = int(len(train) - n_bcc_train)
assert n_bcc_train == 3323, f"§3: BCC en train: esperado 3323, obtenido {n_bcc_train}"
assert n_nobcc_train == 22008, f"§3: no-BCC en train: esperado 22008, obtenido {n_nobcc_train}"

pct_bcc_train = n_bcc_train / len(train) * 100
pct_nobcc_train = n_nobcc_train / len(train) * 100
assert abs(pct_bcc_train - 13.12) < 0.01, (
    f"§3: % BCC en train: esperado 13,12 %, obtenido {pct_bcc_train:.2f} %"
)
assert abs(pct_nobcc_train - 86.88) < 0.01, (
    f"§3: % no-BCC en train: esperado 86,88 %, obtenido {pct_nobcc_train:.2f} %"
)

# Desbalance de la tarea binaria: 1 BCC por cada ~6,6 no-BCC
ratio_desbalance = n_nobcc_train / n_bcc_train
assert abs(ratio_desbalance - 6.6) < 0.1, (
    f"§3: desbalance 1:6,6 esperado (±0,1); obtenido 1:{ratio_desbalance:.2f}"
)

# --- Test evaluable: se excluyen las 2 047 UNK porque no son evaluables (§4)
n_unk_test = int((test["clase_9"] == "UNK").sum())
assert n_unk_test == 2047, f"§4: UNK en test: esperadas 2047, obtenidas {n_unk_test}"

test_evaluable = test[test["clase_9"] != "UNK"].copy()
assert len(test_evaluable) == 6191, (
    f"§4: pool evaluable del test: esperadas 6191 imágenes, obtenidas {len(test_evaluable)}"
)

n_bcc_test = int(test_evaluable["es_bcc"].sum())
n_nobcc_test = int(len(test_evaluable) - n_bcc_test)
assert n_bcc_test == 975, f"§4: BCC en test evaluable: esperado 975, obtenido {n_bcc_test}"
assert n_nobcc_test == 5216, f"§4: no-BCC en test evaluable: esperado 5216, obtenido {n_nobcc_test}"

pct_bcc_test = n_bcc_test / len(test_evaluable) * 100
pct_nobcc_test = n_nobcc_test / len(test_evaluable) * 100
assert abs(pct_bcc_test - 15.75) < 0.01, (
    f"§4: % BCC en test evaluable: esperado 15,75 %, obtenido {pct_bcc_test:.2f} %"
)
assert abs(pct_nobcc_test - 84.25) < 0.01, (
    f"§4: % no-BCC en test evaluable: esperado 84,25 %, obtenido {pct_nobcc_test:.2f} %"
)

vista_binaria = pd.DataFrame(
    [
        {"particion": "train", "grupo": "BCC", "n": n_bcc_train, "pct": round(pct_bcc_train, 2)},
        {"particion": "train", "grupo": "no-BCC", "n": n_nobcc_train, "pct": round(pct_nobcc_train, 2)},
        {"particion": "test_evaluable", "grupo": "BCC", "n": n_bcc_test, "pct": round(pct_bcc_test, 2)},
        {"particion": "test_evaluable", "grupo": "no-BCC", "n": n_nobcc_test, "pct": round(pct_nobcc_test, 2)},
    ]
)

print(f"[§3] Desbalance binario del train: 1 : {ratio_desbalance:.2f} (BCC : no-BCC)")
print("[§3, §4] Vista binaria de la tarea (train completo y test evaluable sin UNK):")
print(vista_binaria.to_string(index=False))

ruta_vista = os.path.join(SALIDAS_DIR, "vista_binaria.csv")
vista_binaria.to_csv(ruta_vista, index=False)
print(f"\n[guardado] {ruta_vista}")


In [ ]:
# =============================================================================
# Celda 6 — Calidad de metadatos: faltantes REALES (§3.1, §4.1)
# pandas ya convierte el literal 'nan' a NaN; aquí se cuentan ambos (NaN y el texto
# explícito) y, en edad, el valor 0 que §3.1 documenta como «edad desconocida».
# =============================================================================
LITERALES_FALTANTES = ["nan", "none", ""]


def contar_faltantes_texto(serie):
    """Cuenta NaN de pandas + literales de texto que representan ausencia ('nan', 'none', '')."""
    n_pandas = int(serie.isna().sum())
    como_texto = serie.astype("string").str.strip().str.lower()
    n_literal = int(como_texto.isin(LITERALES_FALTANTES).sum())
    return n_pandas + n_literal


def resumen_calidad(metadata, particion):
    """Devuelve una fila por campo con el faltante real, su % y la regla aplicada."""
    filas = []
    for campo in ("anatom_site_general", "sex", "age_approx"):
        n_faltantes = contar_faltantes_texto(metadata[campo])
        regla = "NaN de pandas o literal 'nan'"
        if campo == "age_approx":
            n_cero = int((metadata[campo] == 0).sum())
            n_faltantes += n_cero
            regla = "NaN, literal 'nan' o age_approx == 0 (edad desconocida)"
        filas.append(
            {
                "particion": particion,
                "campo": campo,
                "faltantes": int(n_faltantes),
                "n_total": int(len(metadata)),
                "pct": round(n_faltantes / len(metadata) * 100, 2),
                "regla": regla,
            }
        )
    return pd.DataFrame(filas)


calidad = pd.concat(
    [resumen_calidad(meta_train, "train"), resumen_calidad(meta_test, "test")],
    ignore_index=True,
)


def faltantes_de(particion, campo):
    """Extrae el número de faltantes de la tabla de calidad."""
    fila = calidad[(calidad["particion"] == particion) & (calidad["campo"] == campo)]
    assert len(fila) == 1, f"Calidad: fila no encontrada para {particion}/{campo}"
    return int(fila["faltantes"].iloc[0])


# §3.1 y §4.1: faltantes reales esperados (conteo y porcentaje)
ESPERADO_FALTANTES = {
    ("train", "anatom_site_general"): (2631, 10.4),
    ("train", "sex"): (384, 1.5),
    ("test", "anatom_site_general"): (655, 8.0),
    ("test", "sex"): (339, 4.1),
}
for (particion, campo), (esperado, pct_esperado) in ESPERADO_FALTANTES.items():
    obtenido = faltantes_de(particion, campo)
    assert obtenido == esperado, (
        f"§3.1/§4.1: faltantes de {campo} en {particion}: esperado {esperado}, obtenido {obtenido}"
    )
    pct_obtenido = float(
        calidad[(calidad["particion"] == particion) & (calidad["campo"] == campo)]["pct"].iloc[0]
    )
    assert abs(pct_obtenido - pct_esperado) <= 0.1, (
        f"§3.1/§4.1: % de faltantes de {campo} en {particion}: "
        f"esperado {pct_esperado} %, obtenido {pct_obtenido} %"
    )

# --- Edad del train: el CSV tiene 54 ceros y 437 NaN (491 edades desconocidas en total).
# §3.1 del análisis redacta «113 edades desconocidas»: discrepancia documentada, se advierte
# y se conserva la cifra reproducible del CSV oficial.
n_cero_train = int((meta_train["age_approx"] == 0).sum())
n_nan_train = int(meta_train["age_approx"].isna().sum())
assert n_cero_train == 54, (
    f"§3.1: edades con age_approx == 0 en train: esperadas 54, obtenidas {n_cero_train}"
)
assert n_nan_train == 437, (
    f"§3.1: edades NaN en train: esperadas 437, obtenidas {n_nan_train}"
)
assert faltantes_de("train", "age_approx") == 491, (
    f"§3.1: edades desconocidas en train: esperadas 491, obtenidas "
    f"{faltantes_de('train', 'age_approx')}"
)
if n_cero_train != 113:
    print(
        "ADVERTENCIA (§3.1): el análisis cita «113 edades desconocidas» con age_approx == 0; "
        f"el CSV oficial contiene {n_cero_train} ceros y {n_nan_train} NaN "
        f"({n_cero_train + n_nan_train} edades desconocidas en total: "
        f"{(n_cero_train + n_nan_train) / len(meta_train) * 100:.2f} %). "
        "Se conserva la cifra reproducible del CSV."
    )

print("[§3.1, §4.1] Calidad de metadatos (faltantes reales, no solo celdas vacías):")
print(calidad.to_string(index=False))

ruta_calidad = os.path.join(SALIDAS_DIR, "calidad_metadatos.csv")
calidad.to_csv(ruta_calidad, index=False)
print(f"\n[guardado] {ruta_calidad}")


In [ ]:
# =============================================================================
# Celda 7 — Análisis de lesion_id en el train (§3.2)
# El campo existe SOLO en el train: el test no lo trae (§4.3).
# =============================================================================
lesion_id = train["lesion_id"]
n_sin_id = int(lesion_id.isna().sum())
n_con_id = int(lesion_id.notna().sum())
assert n_con_id == 23247, f"§3.2: imágenes con lesion_id: esperadas 23247, obtenidas {n_con_id}"
assert n_sin_id == 2084, f"§3.2: imágenes sin lesion_id: esperadas 2084, obtenidas {n_sin_id}"

pct_con_id = n_con_id / len(train) * 100
assert abs(pct_con_id - 91.8) < 0.1, (
    f"§3.2: % de imágenes con lesion_id: esperado 91,8 %, obtenido {pct_con_id:.2f} %"
)

# Conteo de imágenes por lesión (solo filas con lesion_id)
conteo_por_lesion = lesion_id.dropna().value_counts()
n_lesiones = int(conteo_por_lesion.size)

# Verificación BLANDA del número de lesiones distintas: §3.1 dice 11 848 y §3.2 dice 11 847
# (discrepancia interna del análisis). No se falla: se advierte y se continúa.
if n_lesiones == 11847:
    print(f"[§3.2] Lesiones distintas: {n_lesiones} (coincide con §3.2).")
else:
    if n_lesiones == 11848:
        print(
            f"ADVERTENCIA: el conteo de lesiones distintas da {n_lesiones}, valor de §3.1, "
            "mientras §3.2 documenta 11847. Discrepancia interna del análisis; se continúa."
        )
    else:
        print(
            f"ADVERTENCIA: el conteo de lesiones distintas da {n_lesiones}; §3.2 documenta "
            "11847 y §3.1 11848. Discrepancia del análisis; se continúa sin fallar."
        )

# Lesiones multi-imagen y su peso
lesiones_multi = conteo_por_lesion[conteo_por_lesion > 1]
n_multi = int(lesiones_multi.size)
n_imgs_multi = int(lesiones_multi.sum())
max_imgs = int(conteo_por_lesion.max())

assert n_multi == 5059, f"§3.2: lesiones multi-imagen: esperadas 5059, obtenidas {n_multi}"
assert n_imgs_multi == 16459, (
    f"§3.2: imágenes en lesiones multi-imagen: esperadas 16459, obtenidas {n_imgs_multi}"
)
assert max_imgs == 31, f"§3.2: máximo de imágenes por lesión: esperado 31, obtenido {max_imgs}"

# Distribución completa de imágenes por lesión
distribucion = (
    conteo_por_lesion.value_counts()
    .sort_index()
    .rename_axis("imagenes_por_lesion")
    .reset_index(name="n_lesiones")
)
distribucion["imagenes_totales"] = (
    distribucion["imagenes_por_lesion"] * distribucion["n_lesiones"]
)

# Verificación BLANDA de la distribución documentada en §3.2
ESPERADO_DISTRIBUCION = {1: 6788, 2: 2505, 3: 1306, 4: 468, 5: 248, 31: 1}
desviaciones = []
for imagenes, esperado in ESPERADO_DISTRIBUCION.items():
    obtenido = int(distribucion.loc[distribucion["imagenes_por_lesion"] == imagenes, "n_lesiones"].sum())
    if obtenido != esperado:
        desviaciones.append(f"{imagenes} img: esperado {esperado}, obtenido {obtenido}")
if desviaciones:
    print("ADVERTENCIA (§3.2): la distribución de imágenes por lesión difiere en: "
          + "; ".join(desviaciones))

print(f"[§3.2] Imágenes con lesion_id: {n_con_id} ({pct_con_id:.1f} %) | sin id: {n_sin_id}")
print(f"[§3.2] Lesiones distintas: {n_lesiones} | multi-imagen: {n_multi} ({n_imgs_multi} imágenes)"
      f" | máximo por lesión: {max_imgs}")
print("[§3.2] Distribución de imágenes por lesión:")
print(distribucion.to_string(index=False))
print(
    "\n[consecuencia metodológica] El split train/validación debe agruparse por lesion_id (§3.2); "
    "en el marco v2 las 2 084 imágenes sin id y las de origen test son lesiones propias."
)

resumen_lesion = pd.DataFrame(
    [
        {"metrica": "imagenes_train", "valor": int(len(train))},
        {"metrica": "imagenes_con_lesion_id", "valor": n_con_id},
        {"metrica": "imagenes_sin_lesion_id", "valor": n_sin_id},
        {"metrica": "pct_con_lesion_id", "valor": round(pct_con_id, 2)},
        {"metrica": "lesiones_distintas", "valor": n_lesiones},
        {"metrica": "lesiones_multi_imagen", "valor": n_multi},
        {"metrica": "imagenes_en_lesiones_multi", "valor": n_imgs_multi},
        {"metrica": "max_imagenes_por_lesion", "valor": max_imgs},
    ]
)

ruta_resumen_lesion = os.path.join(SALIDAS_DIR, "lesion_id_resumen.csv")
ruta_distribucion = os.path.join(SALIDAS_DIR, "lesion_id_distribucion.csv")
resumen_lesion.to_csv(ruta_resumen_lesion, index=False)
distribucion.to_csv(ruta_distribucion, index=False)
print(f"\n[guardado] {ruta_resumen_lesion}")
print(f"[guardado] {ruta_distribucion}")


In [ ]:
# =============================================================================
# Celda 8 — Demografía por clase en el train (§7.1–§7.3)
# =============================================================================
CLASES_TRAIN = [c for c in CLASES if c != "UNK"]

# --- (a) Edad por clase: se excluyen los valores no informativos (NaN y age_approx == 0)
edad_valida = train[train["age_approx"].notna() & (train["age_approx"] != 0)]
edad_por_clase = edad_valida.groupby("clase_9")["age_approx"].agg(
    n_con_edad="count", edad_media_sin_cero="mean", edad_mediana_sin_cero="median"
)

# §7.1 se calculó SIN excluir los ceros (los trata como edad 0): se reproduce aparte
# para poder verificar los valores publicados en el análisis.
edad_sin_nan = train[train["age_approx"].notna()]
edad_por_clase_71 = edad_sin_nan.groupby("clase_9")["age_approx"].agg(
    edad_media_con_cero_71="mean", edad_mediana_con_cero_71="median"
)

demografia = (
    train.groupby("clase_9")
    .size()
    .rename("n_total")
    .to_frame()
    .join(edad_por_clase, how="left")
    .join(edad_por_clase_71, how="left")
    .reindex(CLASES_TRAIN)
    .reset_index()
    .rename(columns={"clase_9": "clase"})
)
for columna in ("edad_media_sin_cero", "edad_mediana_sin_cero",
                "edad_media_con_cero_71", "edad_mediana_con_cero_71"):
    demografia[columna] = demografia[columna].round(2)

# §7.1: edad media esperada por clase (tolerancia ±0,3; verificación blanda)
ESPERADO_EDAD_71 = {
    "SCC": (70.8, 70),
    "AK": (67.3, 70),
    "BCC": (65.7, 70),
    "BKL": (63.7, 65),
    "MEL": (60.7, 60),
    "VASC": (53.2, 55),
    "DF": (51.7, 50),
    "NV": (44.9, 45),
}
for clase, (media_esperada, mediana_esperada) in ESPERADO_EDAD_71.items():
    fila = demografia[demografia["clase"] == clase].iloc[0]
    media_obtenida = float(fila["edad_media_con_cero_71"])
    if abs(media_obtenida - media_esperada) > 0.3:
        print(
            f"ADVERTENCIA (§7.1): edad media de {clase}: esperada {media_esperada}, "
            f"obtenida {media_obtenida:.2f} (±0,3)"
        )
    mediana_obtenida = float(fila["edad_mediana_con_cero_71"])
    if abs(mediana_obtenida - mediana_esperada) > 0.5:
        print(
            f"ADVERTENCIA (§7.1): edad mediana de {clase}: esperada {mediana_esperada}, "
            f"obtenida {mediana_obtenida:.1f}"
        )

print("[§7.1] Edad por clase en el train (columnas *_sin_cero excluyen age_approx == 0;")
print("       las columnas *_con_cero_71 reproducen el cálculo publicado en §7.1):")
print(demografia.to_string(index=False))
print("NOTA: §7.1 no excluyó los ceros; con la exclusión estricta (0 = edad desconocida)")
print("      las medias bajan en las clases con más ceros (p. ej. VASC 54,10 vs 53,24).")

# --- (b) Sexo por clase: % dentro de la clase (denominador = total de la clase, §7.2)
sexo = pd.crosstab(train["clase_9"], train["sex"], dropna=False).reindex(CLASES_TRAIN)
sexo.columns = ["sin_dato" if pd.isna(c) else str(c) for c in sexo.columns]
sexo.index.name = "clase"
sexo.columns.name = "sexo"
totales_sexo = sexo.sum(axis=1)
sexo_pct = (sexo.div(totales_sexo, axis=0) * 100).round(2)

assert "male" in sexo_pct.columns, f"§7.2: se esperaba la categoría 'male'; hay {list(sexo_pct.columns)}"
pct_male_bcc = float(sexo_pct.loc["BCC", "male"])
assert abs(pct_male_bcc - 60.2) <= 0.5, (
    f"§7.2: % masculino en BCC: esperado 60,2 % (±0,5), obtenido {pct_male_bcc:.2f} %"
)

sexo_largo = sexo.stack().rename("n").reset_index()
sexo_largo["pct"] = (sexo_largo["n"] / sexo_largo["clase"].map(totales_sexo) * 100).round(2)

print("\n[§7.2] Sexo por clase (% dentro de la clase):")
print(sexo_pct.to_string())

# --- (c) Sitio anatómico por clase: % dentro de la clase (§7.3)
sitio = pd.crosstab(train["clase_9"], train["anatom_site_general"], dropna=False).reindex(CLASES_TRAIN)
sitio.columns = ["sin_dato" if pd.isna(c) else str(c) for c in sitio.columns]
sitio.index.name = "clase"
sitio.columns.name = "sitio"
totales_sitio = sitio.sum(axis=1)
sitio_pct = (sitio.div(totales_sitio, axis=0) * 100).round(2)

pct_cabeza_bcc = float(sitio_pct.loc["BCC", "head/neck"])
pct_torso_bcc = float(sitio_pct.loc["BCC", "anterior torso"])
assert abs(pct_cabeza_bcc - 34.0) <= 0.5, (
    f"§7.3: % cabeza/cuello en BCC: esperado 34,0 % (±0,5), obtenido {pct_cabeza_bcc:.2f} %"
)
assert abs(pct_torso_bcc - 33.5) <= 0.5, (
    f"§7.3: % torso anterior en BCC: esperado 33,5 % (±0,5), obtenido {pct_torso_bcc:.2f} %"
)

sitio_largo = sitio.stack().rename("n").reset_index()
sitio_largo["pct"] = (sitio_largo["n"] / sitio_largo["clase"].map(totales_sitio) * 100).round(2)

print("\n[§7.3] Sitio anatómico por clase (% dentro de la clase):")
print(sitio_pct.to_string())
print(
    "\n[lectura §7] BCC: 65,7 años, 60,2 % hombres, 34,0 % cabeza/cuello y 33,5 % torso anterior.\n"
    "Son señales clínicas reales (fotodaño, edad) y a la vez ATAJOS potenciales si el modelo\n"
    "aprende del fondo o del metadato en lugar de la lesión: el modelo solo recibirá píxel."
)

ruta_demografia = os.path.join(SALIDAS_DIR, "demografia_por_clase.csv")
ruta_sexo = os.path.join(SALIDAS_DIR, "sexo_por_clase.csv")
ruta_sitio = os.path.join(SALIDAS_DIR, "sitio_por_clase.csv")
demografia.to_csv(ruta_demografia, index=False)
sexo_largo.to_csv(ruta_sexo, index=False)
sitio_largo.to_csv(ruta_sitio, index=False)
print(f"\n[guardado] {ruta_demografia}")
print(f"[guardado] {ruta_sexo}")
print(f"[guardado] {ruta_sitio}")


In [ ]:
# =============================================================================
# Celda 9 — Proporción train/test por clase (§5)
# =============================================================================
proporciones = pd.DataFrame(
    {
        "clase": CLASES,
        "n_train": [int(comp_train.get(c, 0)) for c in CLASES],
        "n_test": [int(comp_test.get(c, 0)) for c in CLASES],
    }
)
total_por_clase = proporciones["n_train"] + proporciones["n_test"]
proporciones["pct_en_train"] = (proporciones["n_train"] / total_por_clase * 100).round(1)
proporciones["pct_en_test"] = (proporciones["n_test"] / total_por_clase * 100).round(1)

# §5: % en train esperado por clase (tolerancia ±0,2; verificación blanda)
ESPERADO_PCT_TRAIN = {
    "MEL": 77.3,
    "NV": 83.8,
    "BCC": 77.3,
    "AK": 69.9,
    "BKL": 79.9,
    "DF": 72.4,
    "VASC": 70.9,
    "SCC": 79.2,
    "UNK": 0.0,
}
for clase, esperado in ESPERADO_PCT_TRAIN.items():
    obtenido = float(proporciones.loc[proporciones["clase"] == clase, "pct_en_train"].iloc[0])
    if abs(obtenido - esperado) > 0.2:
        print(
            f"ADVERTENCIA (§5): % en train de {clase}: esperado {esperado} %, "
            f"obtenido {obtenido} % (±0,2)"
        )

print("[§5] Proporción train/test por clase:")
print(proporciones.to_string(index=False))

pct_train_global = len(train) / (len(train) + len(test)) * 100
print(
    f"\n[§5] Conclusión: la partición oficial es aproximadamente 3:1 (75/25) en todas las "
    f"clases etiquetadas; en el total, {pct_train_global:.1f} % de las imágenes son de train. "
    "El UNK solo existe en el test."
)

ruta_proporciones = os.path.join(SALIDAS_DIR, "proporciones_train_test.csv")
proporciones.to_csv(ruta_proporciones, index=False)
print(f"\n[guardado] {ruta_proporciones}")


In [ ]:
# =============================================================================
# Celda 10 — Verificación de fuga por identificador train ↔ test (§6)
# =============================================================================
ids_train = set(train["image"])
ids_test = set(test["image"])
ids_comunes = sorted(ids_train & ids_test)

assert len(ids_train) == 25331, (
    f"§6: ids únicos en train: esperados 25331, obtenidos {len(ids_train)}"
)
assert len(ids_test) == 8238, f"§6: ids únicos en test: esperados 8238, obtenidos {len(ids_test)}"
assert len(ids_comunes) == 0, (
    f"§6: fuga por identificador: esperados 0 ids comunes train↔test, obtenidos "
    f"{len(ids_comunes)} (ejemplos: {ids_comunes[:5]})"
)

# La verificación de CONTENIDO (MD5 idénticos y casi-duplicados pHash) requiere las imágenes
# y ya fue realizada en el análisis (§6): 0 MD5 cruzados, 50 duplicados dentro del train y
# 3 casi-duplicados en test ya excluidos del diseño v2.
fuga = {
    "ids_unicos_train": len(ids_train),
    "ids_unicos_test": len(ids_test),
    "ids_comunes_train_test": len(ids_comunes),
    "ejemplos_ids_comunes": ids_comunes[:5],
    "verificacion_contenido": (
        "Requiere imágenes y ya fue realizada en §6: 0 MD5 idénticos train↔test, "
        "50 duplicados exactos dentro del train y 3 casi-duplicados pHash del test "
        "(ISIC_0034511, ISIC_0034619, ISIC_0068690) ya excluidos en el diseño del marco v2."
    ),
    "fuga_por_lesion_id": (
        "No verificable: el test oficial no trae lesion_id (§4.3, limitación estructural)."
    ),
}

print(f"[§6] IDs únicos: train {len(ids_train)} | test {len(ids_test)} | comunes {len(ids_comunes)}")
print("[§6] Sin fuga por identificador. La verificación de contenido (MD5/pHash) ya se hizo")
print("     sobre las imágenes en el análisis: 0 MD5 cruzados y 3 casi-duplicados excluidos.")

ruta_fuga = os.path.join(SALIDAS_DIR, "fuga_ids.json")
with open(ruta_fuga, "w", encoding="utf-8") as archivo:
    json.dump(fuga, archivo, ensure_ascii=False, indent=2)
print(f"\n[guardado] {ruta_fuga}")


In [ ]:
# =============================================================================
# Celda 11 — Guardado del resumen estructurado y gráficos opcionales
# =============================================================================
conclusiones = [
    "1) El dataset tiene 33 569 imágenes íntegras (25 331 train + 8 238 test) y 0 corruptas (§1).",
    "2) Las 9 clases vienen en one-hot sin multi-etiqueta ni filas vacías: la conversión a "
    "clase única es exacta (§2).",
    "3) Las 2 047 imágenes UNK del test no son evaluables: el pool evaluable es 6 191 (§4).",
    "4) El desbalance binario es 1:6,6 (3 323 BCC vs 22 008 no-BCC): el entrenamiento exige "
    "balanceo (marco v2: undersampling 1:1) y la accuracy sola no es métrica válida (§3).",
    "5) lesion_id solo existe en el train: 5 059 lesiones multi-imagen (máximo 31) obligan a "
    "partir por lesión; las 2 084 imágenes sin id son lesiones propias (§3.2).",
    "6) El test no trae lesion_id: limitación estructural del dataset oficial (§4.3).",
    "7) No hay fuga por contenido entre train y test (0 ids comunes; 0 MD5 cruzados) (§6).",
    "8) El BCC se concentra en 65,7 años, 60,2 % hombres y 34,0 % cabeza/cuello: señal clínica "
    "real y riesgo de atajo; el modelo solo recibirá píxel (§7).",
    "9) Hay dos familias de resolución (1024×1024 y 600×450): el preprocesamiento debe "
    "redimensionar siempre (§3.3).",
    "10) El techo de BCC al fusionar los pools es 4 298: base del marco muestral v2 "
    "(model/docs/marco_muestral_v2.md); §8.4, §8.5 y el anexo del análisis están obsoletos.",
]

resumen_eda = {
    "meta": {
        "notebook": "02_analisis_exploratorio.ipynb",
        "fase_crisp_dm": "Fase 2 — Entendimiento de los datos (Sprint 1)",
        "semilla": SEED,
        "base_dir": BASE_DIR,
        "fuente_datos": "4 CSV oficiales de ISIC 2019 (sin ZIP de imágenes)",
        "referencia_vigente": "PRODUCTO/DATA/ANALISIS_DATASET_ISIC2019.md §1–§7",
        "referencia_marco": "PRODUCTO/DERM_IAPP/model/docs/marco_muestral_v2.md",
        "nota_vigencia": "§8.4, §8.5 y anexo del análisis = marco v1 (obsoleto)",
        "n_train": int(len(train)),
        "n_test": int(len(test)),
    },
    "composicion": {
        fila["clase"]: {"n_train": int(fila["n_train"]), "n_test": int(fila["n_test"])}
        for _, fila in composicion.iterrows()
    },
    "binaria": {
        "train_bcc": n_bcc_train,
        "train_no_bcc": n_nobcc_train,
        "train_pct_bcc": round(pct_bcc_train, 2),
        "train_desbalance_no_bcc_por_bcc": round(ratio_desbalance, 2),
        "test_unk": n_unk_test,
        "test_pool_evaluable": int(len(test_evaluable)),
        "test_bcc": n_bcc_test,
        "test_no_bcc": n_nobcc_test,
        "test_pct_bcc": round(pct_bcc_test, 2),
    },
    "calidad_metadatos": calidad.to_dict(orient="records"),
    "lesion_id": {
        "imagenes_con_id": n_con_id,
        "imagenes_sin_id": n_sin_id,
        "pct_con_id": round(pct_con_id, 2),
        "lesiones_distintas": n_lesiones,
        "lesiones_multi_imagen": n_multi,
        "imagenes_en_lesiones_multi": n_imgs_multi,
        "max_imagenes_por_lesion": max_imgs,
        "distribucion": distribucion.to_dict(orient="records"),
    },
    "demografia_bcc": {
        "edad_media_sin_cero": float(
            demografia.loc[demografia["clase"] == "BCC", "edad_media_sin_cero"].iloc[0]
        ),
        "edad_mediana_sin_cero": float(
            demografia.loc[demografia["clase"] == "BCC", "edad_mediana_sin_cero"].iloc[0]
        ),
        "pct_masculino": round(pct_male_bcc, 2),
        "pct_cabeza_cuello": round(pct_cabeza_bcc, 2),
        "pct_torso_anterior": round(pct_torso_bcc, 2),
    },
    "fuga": fuga,
    "proporciones_train_test": proporciones.to_dict(orient="records"),
    "conclusiones": conclusiones,
}

ruta_resumen_json = os.path.join(SALIDAS_DIR, "resumen_eda.json")
with open(ruta_resumen_json, "w", encoding="utf-8") as archivo:
    json.dump(resumen_eda, archivo, ensure_ascii=False, indent=2)
print(f"[guardado] {ruta_resumen_json}")

# --- Gráficos OPCIONALES: nunca deben romper el notebook si falta matplotlib
if HAY_MATPLOTLIB:
    try:
        # 1) Composición de las 9 clases (train vs test)
        figura, eje = plt.subplots(figsize=(10, 5))
        posiciones = np.arange(len(composicion))
        eje.bar(posiciones - 0.2, composicion["n_train"], width=0.4, label="train")
        eje.bar(posiciones + 0.2, composicion["n_test"], width=0.4, label="test")
        eje.set_xticks(posiciones)
        eje.set_xticklabels(composicion["clase"])
        eje.set_xlabel("Clase ISIC 2019")
        eje.set_ylabel("Nº de imágenes")
        eje.set_title("Composición de las 9 clases (train vs test) — §3, §4")
        eje.legend()
        figura.tight_layout()
        figura.savefig(os.path.join(SALIDAS_DIR, "01_composicion_9_clases.png"), dpi=120)
        plt.close(figura)

        # 2) Vista binaria BCC / no-BCC (train y test evaluable)
        figura, eje = plt.subplots(figsize=(7, 5))
        etiquetas = [
            "train BCC", "train no-BCC", "test eval. BCC", "test eval. no-BCC",
        ]
        valores = [n_bcc_train, n_nobcc_train, n_bcc_test, n_nobcc_test]
        eje.bar(etiquetas, valores, color=["#b2182b", "#4393c3", "#ef8a62", "#92c5de"])
        for indice, valor in enumerate(valores):
            eje.text(indice, valor, f"{valor}", ha="center", va="bottom")
        eje.set_ylabel("Nº de imágenes")
        eje.set_title("Vista binaria BCC / no-BCC — §3, §4")
        figura.tight_layout()
        figura.savefig(os.path.join(SALIDAS_DIR, "02_vista_binaria.png"), dpi=120)
        plt.close(figura)

        # 3) Edad media por clase (excluyendo age_approx == 0)
        orden = demografia.sort_values("edad_media_sin_cero", ascending=False)
        figura, eje = plt.subplots(figsize=(8, 5))
        eje.bar(orden["clase"], orden["edad_media_sin_cero"], color="#5aae61")
        for indice, (clase, media) in enumerate(
            zip(orden["clase"], orden["edad_media_sin_cero"])
        ):
            eje.text(indice, media, f"{media:.1f}", ha="center", va="bottom")
        eje.set_xlabel("Clase")
        eje.set_ylabel("Edad media (años)")
        eje.set_title("Edad media por clase en el train (sin age_approx == 0) — §7.1")
        figura.tight_layout()
        figura.savefig(os.path.join(SALIDAS_DIR, "03_edad_media_por_clase.png"), dpi=120)
        plt.close(figura)
        print("[graficos] 3 gráficos PNG generados en salidas_eda/")
    except Exception as error:  # los gráficos son opcionales: nunca rompen el notebook
        print(f"ADVERTENCIA: no se pudieron generar los gráficos ({error}). El EDA continúa.")
else:
    print("[graficos] matplotlib no disponible: se omiten los 3 gráficos (no es un error).")

# --- Listado de archivos generados
print("\n[salidas] Archivos generados en salidas_eda/:")
for nombre_archivo in sorted(os.listdir(SALIDAS_DIR)):
    ruta_archivo = os.path.join(SALIDAS_DIR, nombre_archivo)
    if os.path.isfile(ruta_archivo):
        print(f"  - {nombre_archivo} ({os.path.getsize(ruta_archivo)} B)")


In [ ]:
# =============================================================================
# Celda 12 — Conclusiones del EDA (Fase 2 CRISP-DM)
# =============================================================================
print("=" * 78)
print("CONCLUSIONES DEL ANÁLISIS EXPLORATORIO — ISIC 2019 (Fase 2 CRISP-DM)")
print("=" * 78)
for conclusion in conclusiones:
    print(conclusion)

print("\n" + "-" * 78)
print("Marco muestral vigente: model/docs/marco_muestral_v2.md (90 subconjuntos TA/NFS/PEO,")
print("techo de BCC = 4 298). §8.4, §8.5 y el anexo del análisis son históricos (marco v1).")
print("-" * 78)
print("TODOS los asserts de calidad pasaron ✓")
